# Stage 2: profiles, segments, churn model
Real vs synthetic columns, segment counts, model vs baseline, one customer card.

In [1]:
import sys, pathlib, json; ROOT = pathlib.Path.cwd().parent; sys.path.insert(0, str(ROOT))
import pandas as pd
dd = pd.read_csv(ROOT/"data/processed/data_dictionary.csv")
dd.groupby(["table","lineage"]).size().unstack(fill_value=0)

lineage,borrowed,derived,real,synthetic
table,,,,
bills,3,1,2,2
contact_history,0,0,1,7
customers,0,2,7,6
edge_cases,0,0,1,1
fairness_attributes,0,0,5,0
monthly_activity,0,0,1,4


In [2]:
from collections import Counter
from src.profile.assembler import assemble_many
from src.segmentation.rules import segment
profiles = assemble_many()
results = [segment(p) for p in profiles]
print("primary segment / outcome:", Counter(r.primary_segment or r.outcome for r in results))
print("all labels (multi-label):  ", Counter(s for r in results for s in r.segments))

primary segment / outcome: Counter({'loyalty_candidate': 2422, 'payment_reminder': 1132, 'insufficient_history': 862, 'consistent': 804, 'declining_activity': 744, 'no_segment': 730, 'dormant': 349})
all labels (multi-label):   Counter({'consistent': 3077, 'loyalty_candidate': 2855, 'payment_reminder': 1132, 'declining_activity': 992, 'dormant': 475})


In [3]:
m = json.loads((ROOT/"models/churn_model_metrics.json").read_text())
pd.DataFrame({k: m[k] for k in ["logistic_regression","lightgbm"]}).T

,roc_auc,precision,recall,f1
logistic_regression,0.9156,0.7672,0.6872,0.7250
lightgbm,0.9480,0.7842,0.7968,0.7905


In [4]:
from src.segmentation.rules import segment_with_model
p = next(p for p in profiles if p.churn_risk and p.churn_risk > 0.7 and p.tenure_months >= 3)
r = segment_with_model(p)
print(f"Customer {p.customer_id} | {p.plan_name} | tenure {p.tenure_months} m | churn risk {p.churn_risk}")
print("segments:", r.segments, "primary:", r.primary_segment, "confidence:", r.confidence)
for rc in r.reason_codes: print(f"  - [{rc.code}] {rc.text}  ({rc.evidence_field} = {rc.value})")

Customer 0004-TLHLJ | Fiber Flex | tenure 4 m | churn risk 0.8306
segments: ['payment_reminder', 'declining_activity'] primary: payment_reminder confidence: 1.0
  - [BILL_OVERDUE] Oldest unpaid bill is 9 days past due  (max_days_past_due = 9)
  - [AMOUNT_OVERDUE] Overdue balance on the account  (overdue_amount = 36.95)
  - [HIGH_CHURN_RISK] Churn-risk score 0.83  (churn_risk = 0.8306)
  - [CHURN_DRIVER] Time as a customer raises churn risk  (tenure_months = 4)
  - [CHURN_DRIVER] Month-to-month contract raises churn risk  (contract = Month-to-month)
  - [CHURN_DRIVER] Has fiber internet raises churn risk  (services = Fiber internet;Phone line;Device protection)
